# Двухфазный агент против обычного v5 (Colab A100, задачи конкурса)

Зачем: в 47% провалов база открывает нужный файл за 30-50 с, называет причину и потом ~17 вызовов читает дальше, не правя.
Двухфазный агент (`sim/two_phase.py`): **следователь** (только чтение, отвечает диагнозом) -> **исправитель**
(промпт v5 + диагноз, история -- только условие). Бюджет общий: как у v5 (60 вызовов / 5,5 мин).
Следователь -- базовая модель (он не правит, адаптер правок ему не нужен), исправитель -- с адаптером `pref_edit_v1`.

Набор `eval/two_phase_small.json`: 46 маленьких задач (до 15 строк в эталоне): 21 решаемая базой «иногда», 17 «никогда»
и 8 контрольных «всегда» (не сломал ли двухфазный режим лёгкие задачи). По 2 повтора каждого режима.

Ячейки: **0** параметры, **1** GPU, **2** код, **3** Drive и v5, **4** модель, **5** vLLM, **6** песочница (ноутбук),
**7** прогоны (~4 ч, результаты сразу в Drive, прерванный прогон продолжается), **8** сравнение.

## 0. Параметры

In [ ]:
SUBMISSION_ZIP = "submissions/v5.zip"     # промпт, лимиты, температура и адаптер берутся отсюда
MODES = ["two_phase", "v5"]               # порядок прогонов; RUN_ONLY в ячейке 7 -- запустить один
REPEATS = 2
WORKERS = 4                               # не больше --max-sessions сервера
OUT_ROOT_NAME = "two_phase_eval"
# следователь не правит файлы, адаптер правок ему не нужен: базовая модель (как планируется на Kaggle);
# исправитель -- с адаптером отправки (pref_edit_v1)
INVESTIGATOR_MODEL = "gemma-4-31b-it-qat-w4a16-ct"
INVESTIGATOR_ADAPTER = None               # например "inv_v1" (папка в gemma_adapters): следователь с этим адаптером

## 1. Карта

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)

## 2. Код и зависимости

In [ ]:
import os
if not os.path.exists("/content/gemma4-swe-rl"):
    !git clone -q https://github.com/aletonfdg/gemma4-swe-rl.git /content/gemma4-swe-rl
%cd /content/gemma4-swe-rl
!git pull -q
!git log --oneline | head -3
!pip install -q vllm openai pyyaml requests huggingface_hub kagglehub
import json
SET = json.load(open("eval/two_phase_small.json", encoding="utf-8"))
IDS = SET["all_ids"]
assert len(IDS) == 46 and os.path.exists("sim/two_phase.py") and os.path.exists("prompts/two_phase/investigator.md")
print("задач:", len(IDS))

## 3. Drive и отправка v5

In [ ]:
import re, shutil, zipfile, yaml
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/gemma_adapters"
OUT_ROOT = f"{DRIVE}/{OUT_ROOT_NAME}"
os.makedirs(OUT_ROOT, exist_ok=True)
d = "/content/subs/v5"
shutil.rmtree(d, ignore_errors=True)
zipfile.ZipFile(f"{DRIVE}/{SUBMISSION_ZIP}").extractall(d)
ROOT = next(p for p, _, files in os.walk(d) if "agent.yaml" in files)
agent = open(f"{ROOT}/agent.yaml", encoding="utf-8").read()
ADAPTER = re.search(r"(?m)^adapter:\s*(\S+)", agent).group(1)
EV = (yaml.safe_load(open(f"{ROOT}/eval_config.yaml")) or {}).get("evaluation", {})
TEMP = (yaml.safe_load(open(f"{ROOT}/configs/sampling.yaml")) or {}).get("temperature", 0.2)
SYSTEM = f"{ROOT}/prompts/system.md"
assert os.path.exists(f"{ROOT}/adapters/{ADAPTER}/adapter_model.safetensors")
print("адаптер", ADAPTER, "| вызовов", EV.get("max_tool_calls"), "| минут", EV.get("max_time_minutes"), "| T", TEMP)

## 4. Базовая модель в `/content/model` (если уже есть, ничего не качает)

In [ ]:
import getpass
MODEL_DIR = "/content/model"
if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    try:
        from huggingface_hub import snapshot_download
        hf_token = getpass.getpass("HF токен (Enter, если не нужен): ").strip() or None
        snapshot_download("google/gemma-4-31b-it-qat-w4a16-ct", local_dir=MODEL_DIR, token=hf_token)
    except Exception as e:
        print("HF не получилось:", type(e).__name__, str(e)[:300])
        import kagglehub
        os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("KAGGLE_API_TOKEN (KGAT_...): ").strip()
        path = kagglehub.model_download("google/gemma-4/transformers/gemma-4-31b-it-qat-w4a16-ct")
        !cp -r "{path}"/. {MODEL_DIR}/
!du -sh {MODEL_DIR}

## 5. vLLM с адаптером v5

In [ ]:
import subprocess, time, requests
BASE = "gemma-4-31b-it-qat-w4a16-ct"
cmd = ["vllm", "serve", MODEL_DIR, "--served-model-name", BASE,
       "--quantization", "compressed-tensors", "--dtype", "bfloat16",
       "--max-model-len", "32768", "--gpu-memory-utilization", "0.92", "--max-num-seqs", str(WORKERS),
       "--enable-auto-tool-choice", "--tool-call-parser", "gemma4", "--reasoning-parser", "gemma4",
       "--default-chat-template-kwargs", '{"enable_thinking": true}', "--port", "8000",
       "--enable-lora", "--max-lora-rank", "64", "--max-loras", "2",
       "--lora-modules", f"{ADAPTER}={ROOT}/adapters/{ADAPTER}"]
if INVESTIGATOR_ADAPTER:      # адаптер следователя из Drive (gemma_adapters/<имя>), подаётся под своим именем
    INV_DIR = f"{DRIVE}/{INVESTIGATOR_ADAPTER}"
    assert os.path.exists(f"{INV_DIR}/adapter_model.safetensors"), f"нет адаптера {INV_DIR}"
    cmd += [f"{INVESTIGATOR_ADAPTER}={INV_DIR}"]
    INVESTIGATOR_MODEL = INVESTIGATOR_ADAPTER
proc = subprocess.Popen(cmd, stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
t0 = time.time()
while time.time() - t0 < 1500 and proc.poll() is None:
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=3).status_code == 200:
            print("vLLM готов за %.0f c" % (time.time() - t0)); break
    except requests.RequestException:
        pass
    time.sleep(5)
if proc.poll() is not None:
    print("vLLM упал:"); print(open("/content/vllm.log").read()[-3000:])
print([m["id"] for m in requests.get("http://127.0.0.1:8000/v1/models").json()["data"]])

## 6. Сервер песочницы на ноутбуке (задачи КОНКУРСА: `--comp`, не `--data`)
В двух окнах PowerShell из папки проекта (Docker Desktop запущен, образы `swebench-*` на месте, ноутбук не засыпает):
```
python -m sim.mimo_server --comp data/comp_full --valid data/benchmark_valid.json --max-sessions 4 --ttl 2400
cloudflared tunnel --url http://127.0.0.1:8765
```

In [ ]:
import getpass, urllib.request
os.environ["SWE_SANDBOX_URL"] = input("Адрес туннеля: ").strip().rstrip("/")
os.environ["SWE_SANDBOX_TOKEN"] = getpass.getpass("Токен сервера: ").strip()
req = urllib.request.Request(os.environ["SWE_SANDBOX_URL"] + "/tasks",
                             headers={"Authorization": "Bearer " + os.environ["SWE_SANDBOX_TOKEN"]})
server_ids = set(json.load(urllib.request.urlopen(req, timeout=30))["result"])
missing = [i for i in IDS if i not in server_ids]
assert not missing, f"сервер не знает {len(missing)} задач: запущен ли он с --comp data/comp_full?"
print("сервер готов, задач:", len(server_ids))

## 7. Прогоны (~2 ч на режим при 4 потоках)
В логе двухфазного режима после каждой задачи строка `следователь: N вызовов, диагноз есть/НЕТ ... | исправитель: ...`.
Если у первых 10 задач диагноза почти никогда нет («НЕТ») -- следователь не останавливается: прервите и напишите мне.

In [ ]:
RUN_ONLY = None          # например ["two_phase"]
for mode in MODES:
    if RUN_ONLY and mode not in RUN_ONLY:
        continue
    run_cmd = " ".join(["python -m sim.run_mimo_baseline --model", ADAPTER, "--temperature", str(TEMP),
                        "--repeats", str(REPEATS), "--workers", str(WORKERS),
                        "--thinking on --no-analyzer --clean-start --system-file", SYSTEM,
                        "--tool-calls", str(EV.get("max_tool_calls", 60)), "--time-minutes", str(EV.get("max_time_minutes", 5.5)),
                        f"--two-phase --investigator-model {INVESTIGATOR_MODEL}" if mode == "two_phase" else "",
                        "--out", f"{OUT_ROOT}/{mode}", "--ids", *IDS])
    print("=" * 30, mode, "=" * 30); print(run_cmd)
    !{run_cmd}

## 8. Сравнение
Главное: доля решённых по группам (иногда / никогда / всегда), время до первой правки исходника, no_patch.
Для двухфазного -- как часто следователь отдал диагноз и сколько вызовов на это ушло.

In [ ]:
import collections
def load(mode):
    p = f"{OUT_ROOT}/{mode}/results.jsonl"
    last = {}
    if os.path.exists(p):
        for l in open(p, encoding="utf-8"):
            if l.strip():
                r = json.loads(l)
                if not r.get("error") and not r.get("crashed"):
                    last[(r["instance_id"], r.get("rep", 0))] = r
    return last

def first_src_edit_s(mode, r):
    """Секунда эпизода, когда сделан первый edit_file/write_file не в /tmp."""
    name = r["instance_id"] + (f"__r{r['rep']}" if r.get("rep") else "") + ".json"
    p = f"{OUT_ROOT}/{mode}/messages/{name}"
    if not os.path.exists(p):
        return None
    msgs = json.load(open(p, encoding="utf-8"))["messages"]
    tools = [e for e in r["timing"]["events"] if e["k"] == "tool"]
    k = 0
    for m in msgs:
        for c in m.get("tool_calls") or []:
            a = c["function"]["arguments"]
            a = json.loads(a) if isinstance(a, str) else a
            if c["function"]["name"] in ("edit_file", "write_file") and not str(a.get("filepath", "")).startswith(("/tmp", "tmp/")):
                return tools[k]["at"] if k < len(tools) else None
            k += 1
    return None

G = SET["groups"]
res = {m: load(m) for m in MODES}
print(f"{'режим':<11}{'эпиз':>6}{'решено':>8}{'иногда':>10}{'никогда':>10}{'всегда':>9}{'no_patch':>10}{'1-я правка, с':>15}")
for m, rows in res.items():
    by = collections.defaultdict(lambda: [0, 0])
    for r in rows.values():
        g = G[r["instance_id"]]; by[g][0] += r["passed"]; by[g][1] += 1
    fe = sorted(x for x in (first_src_edit_s(m, r) for r in rows.values()) if x is not None)
    nop = sum((r.get("result") or {}).get("reason") == "no_patch" for r in rows.values())
    cell = lambda g: f"{by[g][0]}/{by[g][1]}"
    print(f"{m:<11}{len(rows):>6}{sum(r['passed'] for r in rows.values()):>8}{cell('sometimes'):>10}{cell('never'):>10}"
          f"{cell('always'):>9}{nop:>10}{(fe[len(fe) // 2] if fe else float('nan')):>15.0f}")
print("база traj_think (без адаптера, 3 прохода) на тех же задачах:",
      sum(sum(v) for v in SET["base_by_task"].values()), "/", sum(len(v) for v in SET["base_by_task"].values()))

tp = [r["two_phase"] for r in res.get("two_phase", {}).values() if r.get("two_phase")]
if tp:
    n = len(tp)
    st = [t for t in tp if t["inv_stopped"]]
    calls = sorted(t["inv_calls"] for t in st)
    print(f"\nследователь: диагноз в {len(st)}/{n} эпизодов; вызовов до диагноза, медиана {calls[len(calls) // 2] if calls else '-'}; "
          f"эпизодов с запрещённым вызовом (на Kaggle это фатально): {sum(t['inv_forbidden'] > 0 for t in tp)}")
    fe = sorted(t["fix_first_edit_call"] for t in tp if t["fix_first_edit_call"])
    print(f"исправитель: правил в {len(fe)}/{n}; первая правка на вызове (медиана) {fe[len(fe) // 2] if fe else '-'}")

common = set(res.get("two_phase", {})) & set(res.get("v5", {}))
if common:
    a = sum(res["two_phase"][k]["passed"] and not res["v5"][k]["passed"] for k in common)
    b = sum(res["v5"][k]["passed"] and not res["two_phase"][k]["passed"] for k in common)
    print(f"\nпарно ({len(common)} эпизодов): решил только двухфазный {a}, только v5 {b}")